# 02 — 운행 시각과 출발 이벤트 따라가기

[English](02_service_days_and_profiles.ipynb) | **한국어**

출발 프로파일은 점 질의를 분 단위로 나열한 화면이 아닙니다. 실습 코드의 실제 이벤트 기반 보존 라벨 범위 탐색을 살펴본 뒤, 의도적으로 느린 오라클과 대조해 보겠습니다.

범위: 검증을 통과한 하나의 운행일, 정수 초, 고정 도보 이동, 도착 시각·탑승 횟수 목적, 대중교통 탑승이 필요한 출발지/목적지 쌍입니다. [06장](../docs/06_departure_profiles_and_reverse_search.ko.md)을 참고하세요.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src" / "raptor.py").is_file():
    raise RuntimeError("Start this notebook from the repository root or notebooks directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from datetime import date
from src import compile_timetable, demo_timetable, rraptor, arrive_by, last_connection, parse_time, format_time
from src.service_time import service_instant
seconds = parse_time("24:06:00")
print("Service seconds:", seconds)
print("Seoul instant:", service_instant(date(2026, 9, 7), seconds).isoformat())
assert seconds == 86760

## 압축된 구간 살펴보기

엔진은 운행 출발 시각에서 접근 도보 시간과 탑승 여유 시간을 빼 출발지 준비 임계값을 구합니다. 임계값을 늦은 시각부터 이른 시각 순으로 처리하고, 각 탑승 라운드의 라벨을 보존합니다.

아래 구간은 양 끝을 포함합니다. 08:00:00의 경계는 다음 구간이 08:00:01에 시작한다는 뜻입니다.

In [ ]:
index = compile_timetable(demo_timetable())
start, end = parse_time("07:59:00"), parse_time("08:12:00")
profile = rraptor(index, "O", "Z", start, end, max_boardings=3, boarding_slack=60)
for segment in profile.segments:
    outcomes = [(format_time(arrival), boardings) for arrival, boardings in segment.signature]
    print(format_time(segment.ready_from), "through", format_time(segment.ready_through), outcomes or "NO_ROUTE")
assert profile.metrics.reused_labels > 0
print("Retained-label reuse:", profile.metrics.reused_labels)

## 느린 방법으로 빠른 방법 확인하기

오라클은 상태 그래프를 탐색하고 탑승 가능한 모든 운행을 열거합니다. 여기서는 작은 시간 구간의 모든 정수 초를 확인합니다. 범위 엔진의 스캔, 마킹, 도보 경로 전파 코드를 재사용하지 않습니다.

이 반복은 테스트에 속하며, 운영 프로파일 엔드포인트나 대체 경로에 속하지 않습니다.

In [ ]:
from src.oracle import exact_profile_seconds
expected = exact_profile_seconds(index.timetable, "O", "Z", start, end, max_boardings=3, boarding_slack=60)
for ready, signature in expected.items():
    journeys = profile.at(ready)
    assert tuple((j.arrival, j.boardings) for j in journeys) == signature
    for journey in journeys:
        journey.validate(60)
print(f"All {len(expected)} integer-second queries agree with the independent oracle.")

## 물리적 보행로가 아니라 질문을 뒤집기

ARRIVE_BY는 목적지 도착 완료 마감 시각 전에 가능한 가장 늦은 출발지 준비 시각을 찾습니다. 역방향 가능성은 기존 정방향 간선의 인덱스를 사용합니다. 반대 물리 방향을 만들어 내지 않습니다.

In [ ]:
deadline = parse_time("08:22:00")
journey = arrive_by(index, "O", "Z", deadline, max_boardings=3, boarding_slack=60)
assert journey is not None
journey.validate(60)
print("Latest ready:", format_time(journey.ready_at))
print("Actual arrival:", format_time(journey.arrival))
assert journey.ready_at == parse_time("08:00:00")
assert arrive_by(index, "Z", "O", deadline, max_boardings=3, boarding_slack=60) is None

## 마지막 연결은 23:59가 아니다

이 작은 아침 시간표에서 가능한 마지막 출발지 출발은 08:10입니다. 결과는 특별한 시각이 아니라 실제로 제공된 이벤트에서 나옵니다. 실제 서비스에서는 운행 달력, 개별 운행편의 식별자, 운행일 범위, 실시간 정보 적용 가능성도 검증해야 합니다.

In [ ]:
last = last_connection(index, "O", "Z", max_boardings=3, boarding_slack=60)
assert last is not None
assert last.ready_at == parse_time("08:10:00")
assert last.arrival == parse_time("08:38:00")
print("Last ready:", format_time(last.ready_at), "Actual arrival:", format_time(last.arrival))

## 확장하기 전에 한계를 명시하기

이 실습은 도보만 사용하는 출발 프로파일을 거부합니다. 그 도착 함수는 상수가 아니라 아핀 함수(`ready + walking_duration`)일 수 있기 때문입니다. 또한 경로 스캔 내부의 여러 날짜 프로파일 병합이나 다기준 대표값도 구현하지 않습니다.

질문: 최소 도보 또는 가장 안전한 연결을 보장하려면 두 구간이 동등해지기 전에 무엇이 바뀌어야 할까요? 답: 중간 프런티어와 구간 서명 모두 그 기준을 보존해야 합니다. 스칼라 목적지 필터는 앞서 버린 경로를 되살릴 수 없습니다.